# 🏆 Power Query Quest — Solution Walkthrough

**Category:** Advanced | **Difficulty:** 🟣 Master

This notebook demonstrates the Power Query transformations using Python and `pandas`.
We import three messy data sources, clean them, unpivot, and merge them into one master dataset.

> In Excel, all of these steps are performed inside the Power Query Editor (Data → Get Data). Here we replicate each transformation in Python so you can follow along.

## Prerequisites

```bash
pip install pandas openpyxl
```

In [ ]:
import pandas as pd
import numpy as np

print('Libraries loaded!')

## Step 1 — Create the Three Messy Data Sources

**Kingdom 1:** Customer records (with nulls and wrong data types)  
**Kingdom 2:** Product price list (with inconsistent column names)  
**Kingdom 3:** Sales log (wide format — months as columns, needs unpivoting)

In [ ]:
# Kingdom 1: Customer records — has nulls and CustomerID as string instead of int
customers_raw = pd.DataFrame({
    'customer_id': ['C001', 'C002', None, 'C004', 'C005', 'C006'],
    'Name': ['Alice', 'Bob', 'Carol', 'Dave', None, 'Frank'],
    'Region': ['North', 'South', 'East', 'West', 'North', 'South'],
    'Joined': ['2024-01-15', '2024-03-22', '2024-05-10', None, '2024-08-01', '2024-09-14']
})

# Kingdom 2: Product price list — column names don't match other tables
products_raw = pd.DataFrame({
    'prod_code': ['P101', 'P102', 'P103', 'P104'],
    'product_name': ['Widget A', 'Widget B', 'Gadget X', 'Gadget Y'],
    'unit_price_usd': [29.99, 49.99, 99.99, 149.99],
    'category': ['Widgets', 'Widgets', 'Gadgets', 'Gadgets']
})

# Kingdom 3: Sales log — wide format (months as columns)
sales_raw = pd.DataFrame({
    'CustomerID': ['C001', 'C002', 'C004', 'C006'],
    'ProductID':  ['P101', 'P103', 'P102', 'P104'],
    'Jan': [5, 2, 8, 1],
    'Feb': [7, 3, 6, 4],
    'Mar': [4, 5, 9, 2],
    'Apr': [None, 4, 7, 3],
    'May': [6, None, 5, 5],
    'Jun': [8, 6, 4, 2]
})

print('Kingdom 1 — Customers (raw):')
print(customers_raw.to_string(index=False))
print()
print('Kingdom 2 — Products (raw):')
print(products_raw.to_string(index=False))
print()
print('Kingdom 3 — Sales Log (raw, wide format):')
print(sales_raw.to_string(index=False))

## Step 2 — Clean Each Query

**Power Query equivalent:**
- Home → Remove Rows → Remove Blank Rows
- Transform → Data Type
- Transform → Rename Columns

In [ ]:
# Clean Kingdom 1: Customers
customers = customers_raw.copy()
customers = customers.dropna(subset=['customer_id', 'Name'])  # Remove rows with null ID or Name
customers = customers.rename(columns={'customer_id': 'CustomerID'})  # Standardize column name
customers['Joined'] = pd.to_datetime(customers['Joined'], errors='coerce')  # Fix data type
customers = customers.dropna(subset=['Joined'])  # Remove rows where date couldn't be parsed

print('Customers (cleaned):')
print(customers.to_string(index=False))

# Clean Kingdom 2: Products
products = products_raw.copy()
products = products.rename(columns={
    'prod_code': 'ProductID',
    'product_name': 'ProductName',
    'unit_price_usd': 'UnitPrice'
})

print()
print('Products (cleaned):')
print(products.to_string(index=False))

## Step 3 — Unpivot the Sales Log

**Power Query equivalent:** Select month columns → Transform → Unpivot Columns

This converts the wide format (months as columns) to long format (one row per month).

In [ ]:
month_cols = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun']

# Unpivot (melt) — equivalent to Power Query's Unpivot Columns
sales_long = sales_raw.melt(
    id_vars=['CustomerID', 'ProductID'],
    value_vars=month_cols,
    var_name='Month',
    value_name='UnitsSold'
)

# Remove nulls that resulted from missing data
sales_long = sales_long.dropna(subset=['UnitsSold'])
sales_long['UnitsSold'] = sales_long['UnitsSold'].astype(int)

print(f'Sales log unpivoted: {len(sales_long)} rows')
print(sales_long.to_string(index=False))

## Step 4 — Merge All Three Queries

**Power Query equivalent:** Home → Merge Queries

We perform two joins:
1. Sales log + Products (on ProductID)
2. Result + Customers (on CustomerID)

In [ ]:
# Join 1: Sales + Products
merged = sales_long.merge(products[['ProductID', 'ProductName', 'UnitPrice', 'category']],
                          on='ProductID', how='left')

# Join 2: Result + Customers
master = merged.merge(customers[['CustomerID', 'Name', 'Region']],
                      on='CustomerID', how='left')

# Add calculated column: Revenue = UnitsSold * UnitPrice
master['Revenue'] = (master['UnitsSold'] * master['UnitPrice']).round(2)

# Reorder columns for clarity
master = master[['CustomerID', 'Name', 'Region', 'ProductID', 'ProductName',
                  'category', 'Month', 'UnitsSold', 'UnitPrice', 'Revenue']]

print(f'Master dataset: {master.shape[0]} rows × {master.shape[1]} columns')
print(master.to_string(index=False))

## Step 5 — Validate the Final Table

**Power Query equivalent:** Close & Load → verify the table in Excel

In [ ]:
# Check for null values
null_counts = master.isnull().sum()

print('Null value check:')
print(null_counts.to_string())
print()

# Summary statistics
print(f'Total Revenue: ${master["Revenue"].sum():,.2f}')
print(f'Top Region: {master.groupby("Region")["Revenue"].sum().idxmax()}')
print(f'Top Product: {master.groupby("ProductName")["Revenue"].sum().idxmax()}')

In [ ]:
# Save the master dataset to Excel — equivalent to Power Query's Close & Load
output_path = '/tmp/power_query_master.xlsx'
master.to_excel(output_path, index=False)
print(f'Master dataset saved to: {output_path}')

## ✅ Challenge Complete!

You've successfully:
- ✅ **Imported** three messy data sources into separate queries
- ✅ **Cleaned** each query: removed nulls, standardized column names, fixed data types
- ✅ **Unpivoted** the sales log from wide to long format
- ✅ **Merged** all three queries into a single master table
- ✅ **Validated** the final output (no nulls, correct totals)

### 🔗 Try Next:
- **[Dashboard Duel](../../data-analysis/dashboard-duel/)** — Build a dashboard from your Power Query master dataset
- **[Macro Mission](../../automation/macro-mission/)** — Automate report formatting with VBA